In [1]:
!pip install -r ../requirements.txt

In [3]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch
import time
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import f1_score, classification_report

import warnings
warnings.filterwarnings('ignore')

DATA_DIR  = "./Data"
MODEL_DIR = "./Models_ASL"

dev_df = pd.read_csv(os.path.join(DATA_DIR, "dev.csv"))
print(f"Dev: {len(dev_df):,}")

start = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
print(f"Model load: {time.time()-start:.1f}s")
print(f"Device: {device}")

with open(os.path.join(MODEL_DIR, "best_threshold.json")) as f:
    best_threshold = json.load(f)['best_threshold']
print(f"Threshold: {best_threshold:.2f}")

def clean_text(text):
    text = str(text)
    text = re.sub(r'(From|To|Cc|Subject|Date|Forwarded by)[^\n]*\n', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

MAX_LENGTH = 512
BATCH_SIZE = 32

class AVDataset(Dataset):
    def __init__(self, df):
        df = df.copy()
        df['text_1'] = df['text_1'].apply(clean_text)
        df['text_2'] = df['text_2'].apply(clean_text)
        self.encodings = tokenizer(
            list(df['text_1']),
            list(df['text_2']),
            max_length=MAX_LENGTH,
            truncation='longest_first',
            padding=False,
            return_tensors=None
        )

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
        }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("✅ Ready!")

Dev: 5,993


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Model load: 5.1s
Device: cuda
Threshold: 0.51
✅ Ready!


In [5]:
start = time.time()
dev_dataset = AVDataset(dev_df)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0,
                        collate_fn=data_collator)
print(f"Tokenizing: {time.time()-start:.1f}s")

start = time.time()
all_probs = []
with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1]
        all_probs.extend(probs.cpu().numpy())

print(f"Inference: {time.time()-start:.1f}s")
final_preds = (np.array(all_probs) >= best_threshold).astype(int)

print(f"F1 Score : {f1_score(dev_df['label'], final_preds):.4f}")
print(f"Macro F1 Score : {f1_score(dev_df['label'], final_preds, average='macro'):.4f}")
print(classification_report(dev_df['label'], final_preds))

Tokenizing: 1.0s
Inference: 355.6s
F1 Score : 0.8365
Macro F1 Score : 0.8348
              precision    recall  f1-score   support

           0       0.83      0.84      0.83      2937
           1       0.84      0.83      0.84      3056

    accuracy                           0.83      5993
   macro avg       0.83      0.83      0.83      5993
weighted avg       0.83      0.83      0.83      5993



In [7]:
demo_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
print(f"Demo: {len(demo_df):,}")

start = time.time()
demo_dataset = AVDataset(demo_df)
demo_loader = DataLoader(demo_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0,
                         collate_fn=data_collator)
print(f"Tokenizing: {time.time()-start:.1f}s")

start = time.time()
all_probs = []
with torch.no_grad():
    for batch in demo_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1]
        all_probs.extend(probs.cpu().numpy())

print(f"Inference: {time.time()-start:.1f}s")
final_preds = (np.array(all_probs) >= best_threshold).astype(int)

pd.DataFrame({'prediction': final_preds}).to_csv("Group_33_C.csv", index=False)
print("✅ Saved → Group_33_C.csv")

Demo: 5,985
Tokenizing: 1.1s
Inference: 370.1s
✅ Saved → Group_33_C.csv


In [9]:
def predict_author_match(text1, text2, model, tokenizer, threshold=0.51):
    t1 = clean_text(text1)
    t2 = clean_text(text2)
    
    inputs = tokenizer(
        t1, t2, 
        max_length=512, 
        truncation='longest_first', 
        padding='max_length', 
        return_tensors='pt'
    ).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        match_prob = probs[0][1].item()
    
    print("\n" + "="*40)
    result_label = "✅ [Same Author]" if match_prob >= threshold else "❌ [Different Author]"
    print(f"  PREDICTION : {result_label}")
    print(f"  PROBABILITY: {match_prob*100:.2f}%")
    print(f"  THRESHOLD  : {threshold*100:.2f}%")
    print("="*40 + "\n")

# Example Usage:
text_a = "I like it. It is good. I want more candy now. Happy today!"
text_b = "seriously though, is it summer yet?"

predict_author_match(text_a, text_b, model, tokenizer, best_threshold)


  PREDICTION : ❌ [Different Author]
  PROBABILITY: 13.45%
  THRESHOLD  : 51.00%

